# Assignment 2 · Embeddings and Hidden Markov Models

CoNLL-2000 POS tagging with a held-out validation split and the original test partition.

In [1]:
import math
from collections import Counter, defaultdict

import numpy as np
from datasets import load_dataset

dataset = load_dataset(
    "parquet",
    data_files={
        "train": "https://huggingface.co/datasets/eriktks/conll2000/resolve/f7bc5ce921058824d5a9843f6a9b6a319e0ac11b/data/train-00000-of-00001.parquet",
        "test": "https://huggingface.co/datasets/eriktks/conll2000/resolve/f7bc5ce921058824d5a9843f6a9b6a319e0ac11b/data/test-00000-of-00001.parquet",
    },
)
train_data = dataset["train"].select_columns(["tokens", "pos_tags"])
indices = np.random.default_rng(42).permutation(len(train_data))
val_size = math.ceil(0.1 * len(train_data))
val_dataset = train_data.select(indices[:val_size].tolist())
train_dataset = train_data.select(indices[val_size:].tolist())
test_dataset = dataset["test"].select_columns(["tokens", "pos_tags"])
print(f"Training: {len(train_dataset):,}; validation: {len(val_dataset):,}; test: {len(test_dataset):,}")

Training: 8,043; validation: 894; test: 2,013


## Part 1 · Word representations

Embedding training lowercases tokens to pool case variants, groups numeric tokens that contain a digit, and retains punctuation. A validation coverage check selects the largest cutoff that sacrifices at most one percentage point of coverage compared with cutoff 2. Rare tokens become `<UNK>`. PPMI uses a two-token window to emphasize local syntactic contexts.

In [2]:
from collections import Counter
import re
# Convert tokens to lowercase and add numbers to <NUM> token 
def normalize_token(token):
    token = token.lower()
    # Require a digit as well as a numeric token shape.
    if any(ch.isdigit() for ch in token) and re.fullmatch(r'[\d,.\-/$%]+', token):
        return "<NUM>"
    return token
# and make <UNK> token for tokens that exist less than 3 times in the training set
token_counts = Counter()
for example in train_dataset:
    for token in example["tokens"]:
        token_counts[normalize_token(token)] += 1
validation_token_counts = Counter(
    normalize_token(token) for row in val_dataset for token in row["tokens"]
)
validation_token_total = sum(validation_token_counts.values())
cutoffs = (2, 3, 5)
coverage = {
    cutoff: sum(count for token, count in validation_token_counts.items()
                if token_counts[token] >= cutoff) / validation_token_total
    for cutoff in cutoffs
}
for cutoff, fraction in coverage.items():
    size = 1 + sum(count >= cutoff for count in token_counts.values())
    print(f"Cutoff {cutoff}: vocabulary {size:,}; validation token coverage {fraction:.2%}")
min_token_count = max(cutoff for cutoff in cutoffs if coverage[cutoff] >= coverage[2] - 0.01)
print(f"Selected cutoff {min_token_count} with at most 1 percentage point of coverage loss.")
vocab = {token for token, count in token_counts.items() if count >= min_token_count}
vocab.add("<UNK>") # Ensure <UNK> is explicitly in our set
def process_sentence(tokens):
    processed = []
    for token in tokens:
        norm_tok = normalize_token(token)
        # Apply <UNK> if the normalized token isn't in our vocab
        if norm_tok in vocab:
            processed.append(norm_tok)
        else:
            processed.append("<UNK>")
    return processed

train_tokens_processed = [process_sentence(example["tokens"]) for example in train_dataset]
vocab_list = sorted(list(vocab))
word2idx = {word: idx for idx, word in enumerate(vocab_list)}
idx2word = {idx: word for idx, word in enumerate(vocab_list)}

# Store the vocabulary size, which is a key hyperparameter for Word2Vec and matrices
vocab_size = len(vocab_list)
print(f"Vocabulary size: {vocab_size}")

val_tokens_processed = [process_sentence(example["tokens"]) for example in val_dataset]

Cutoff 2: vocabulary 7,921; validation token coverage 94.10%
Cutoff 3: vocabulary 5,601; validation token coverage 92.18%
Cutoff 5: vocabulary 3,690; validation token coverage 89.22%
Selected cutoff 2 with at most 1 percentage point of coverage loss.
Vocabulary size: 7921


In [3]:
from collections import defaultdict
import math

def build_ppmi(sentences, window_size, min_pair_count=1):
    co_occurrence_counts = defaultdict(lambda: defaultdict(int))
    total = 0
    for tokens in sentences:
        for i, w in enumerate(tokens):
            for j in range(max(0, i - window_size), min(len(tokens), i + window_size + 1)):
                if j != i:
                    co_occurrence_counts[w][tokens[j]] += 1
                    total += 1
    word_total = {w: sum(c.values()) for w, c in co_occurrence_counts.items()}
    ppmi = defaultdict(dict)
    for w, ctx in co_occurrence_counts.items():
        for c, n in ctx.items():
            if n < min_pair_count:
                continue
            pmi = math.log2((n / total) / ((word_total[w] / total) * (word_total[c] / total)))
            if pmi > 0:
                ppmi[w][c] = pmi
    return ppmi, co_occurrence_counts

ppmi_w2, co_w2 = build_ppmi(train_tokens_processed, window_size=2)
# PMI differs from raw counts because it penalizes words just have high frequencies.

In [4]:
import math
from collections import Counter, defaultdict

def build_TF_IDF(sentences, vocab):
    tf_raw = defaultdict(dict)
    df = defaultdict(int)
    N = len(sentences)

    # 1. Calculate Sublinear TF and DF in a single pass
    for doc_id, tokens in enumerate(sentences):
        token_counts = Counter(tokens)
        
        for token, count in token_counts.items():
            if token in vocab:
                # Sublinear TF: 1 + log(count)
                tf_raw[token][doc_id] = 1 + math.log(count)
                # Document Frequency: count goes up by 1 because it appeared in this doc
                df[token] += 1

    # 2. Calculate IDF and the final TF-IDF sparse matrix
    tf_idf = defaultdict(dict)
    
    for token, doc_dict in tf_raw.items():
        # The row is populated only for words with positive document frequency.
        idf_score = math.log(N / (df[token]))
        
        for doc_id, tf_score in doc_dict.items():
            tf_idf[token][doc_id] = tf_score * idf_score

    return tf_idf

tfidf_sparse = build_TF_IDF(train_tokens_processed, vocab)

### Skip-Gram with Negative Sampling

For target vector $v$, positive context $u_0$, and sampled negative contexts $u_1,\ldots,u_K$, let $s_j=u_j^\top v$, $y_0=1$, and $y_{1:K}=0$. With $\sigma(s)=1/(1+e^{-s})$, the per-pair loss is

$$L=\sum_{j=0}^K[-y_j\log\sigma(s_j)-(1-y_j)\log(1-\sigma(s_j))]
=\sum_{j=0}^K[\operatorname{softplus}(s_j)-y_js_j].$$

Since $\sigma'(s)=\sigma(s)(1-\sigma(s))$, differentiation gives $\partial L/\partial s_j=\sigma(s_j)-y_j$. Applying the chain rule to each dot product yields

$$\frac{\partial L}{\partial v}=\sum_j(\sigma(s_j)-y_j)u_j,\qquad
\frac{\partial L}{\partial u_j}=(\sigma(s_j)-y_j)v.$$

Each minibatch computes all gradients from the same parameter values. SGD subtracts the learning rate times the gradient of the minibatch's summed loss. Repeated IDs accumulate every gradient contribution. The negative-sampling distribution is training unigram frequency raised to $0.75$. A shared validation objective uses the same pairs and random negatives for all configurations.

In [5]:
def skipgram_pairs(sentences, window_size):
    pairs = []
    for tokens in sentences:
        ids = [word2idx[token] for token in tokens]
        for i, center in enumerate(ids):
            start = max(0, i - window_size)
            stop = min(len(ids), i + window_size + 1)
            pairs.extend((center, ids[j]) for j in range(start, stop) if j != i)
    return np.asarray(pairs, dtype=np.int32).reshape(-1, 2)


def train_word2vec(pairs, neg_probs, embedding_dim, num_neg, lr, epochs, seed=42):
    rng = np.random.default_rng(seed)
    target = rng.uniform(-0.5 / embedding_dim, 0.5 / embedding_dim,
                         (len(neg_probs), embedding_dim))
    context = np.zeros_like(target)
    batch_size = 64
    for epoch in range(epochs):
        total_loss = 0.0
        order = rng.permutation(len(pairs))
        for start in range(0, len(order), batch_size):
            centers, positive = pairs[order[start:start + batch_size]].T
            negatives = rng.choice(len(neg_probs), (len(centers), num_neg), p=neg_probs)
            context_ids = np.column_stack((positive, negatives))
            center_vecs = target[centers]
            context_vecs = context[context_ids]
            logits = np.einsum("bd,bkd->bk", center_vecs, context_vecs)
            labels = np.zeros_like(logits)
            labels[:, 0] = 1
            signed_logits = (1 - 2 * labels) * logits
            total_loss += np.logaddexp(0, signed_logits).sum()
            errors = np.exp(-np.logaddexp(0, -logits)) - labels
            target_grad = np.einsum("bk,bkd->bd", errors, context_vecs)
            context_grad = errors[:, :, None] * center_vecs[:, None, :]
            np.add.at(target, centers, -lr * target_grad)
            np.add.at(context, context_ids, -lr * context_grad)
        print(f"Epoch {epoch + 1}/{epochs}, mean training loss {total_loss / len(pairs):.4f}")
    return target, context


def validation_sgns_loss(pairs, target, context, neg_probs, num_neg=5):
    rng = np.random.default_rng(42)
    total = 0.0
    for start in range(0, len(pairs), 512):
        centers, positive = pairs[start:start + 512].T
        negatives = rng.choice(len(neg_probs), (len(centers), num_neg), p=neg_probs)
        v = target[centers]
        positive_scores = np.einsum("bd,bd->b", v, context[positive])
        negative_scores = np.einsum("bd,bkd->bk", v, context[negatives])
        total += np.logaddexp(0, -positive_scores).sum() + np.logaddexp(0, negative_scores).sum()
    return total / len(pairs)


frequencies = np.zeros(len(vocab_list))
for sentence in train_tokens_processed:
    for token in sentence:
        frequencies[word2idx[token]] += 1
neg_probs = frequencies ** 0.75
neg_probs /= neg_probs.sum()
validation_pairs = skipgram_pairs(val_tokens_processed, 2)
configs = [
    dict(window=2, embedding_dim=50, num_neg=5, lr=0.025, epochs=5),
    dict(window=2, embedding_dim=32, num_neg=5, lr=0.04, epochs=3),
    dict(window=3, embedding_dim=50, num_neg=8, lr=0.025, epochs=4),
]
best_loss = np.inf
for config in configs:
    print(config)
    params = {key: value for key, value in config.items() if key != "window"}
    target, context = train_word2vec(skipgram_pairs(train_tokens_processed, config["window"]), neg_probs, **params)
    loss = validation_sgns_loss(validation_pairs, target, context, neg_probs)
    print(f"Validation loss with fixed window=2 and 5 negatives: {loss:.4f}\n")
    if loss < best_loss:
        best_loss, best_config = loss, config
        W_target, W_context = target, context
print(f"Selected {best_config}, validation loss {best_loss:.4f}")
norms = np.linalg.norm(W_target, axis=1, keepdims=True)
W_norm = W_target / np.maximum(norms, 1e-12)

{'window': 2, 'embedding_dim': 50, 'num_neg': 5, 'lr': 0.025, 'epochs': 5}
Epoch 1/5, mean training loss 2.9772
Epoch 2/5, mean training loss 2.4795
Epoch 3/5, mean training loss 2.3667
Epoch 4/5, mean training loss 2.3008
Epoch 5/5, mean training loss 2.2521
Validation loss with fixed window=2 and 5 negatives: 2.2974

{'window': 2, 'embedding_dim': 32, 'num_neg': 5, 'lr': 0.04, 'epochs': 3}
Epoch 1/3, mean training loss 2.8177
Epoch 2/3, mean training loss 2.4072
Epoch 3/3, mean training loss 2.3203
Validation loss with fixed window=2 and 5 negatives: 2.3134

{'window': 3, 'embedding_dim': 50, 'num_neg': 8, 'lr': 0.025, 'epochs': 4}
Epoch 1/4, mean training loss 3.3758
Epoch 2/4, mean training loss 2.8668
Epoch 3/4, mean training loss 2.7964
Epoch 4/4, mean training loss 2.7510
Validation loss with fixed window=2 and 5 negatives: 2.3644

Selected {'window': 2, 'embedding_dim': 50, 'num_neg': 5, 'lr': 0.025, 'epochs': 5}, validation loss 2.2974


### Nearest-neighbor comparison

PPMI downweights co-occurrences expected from common marginals. In these TF-IDF word–document rows, IDF is a per-word scalar and cancels under cosine, so the ranking reflects sublinear sentence co-occurrence. The sparse storage estimates price one value and column index per stored entry. The dense target matrix gives the size of the exported Word2Vec vectors.

In [6]:
import numpy as np

# Build row-normalized vector lookup views, without converting sparse matrices to dense.
def sparse_cosine_neighbors(vectors, query, k=5):
    q = vectors.get(query, {})
    q_norm = math.sqrt(sum(value * value for value in q.values()))
    if q_norm == 0:
        return []
    results = []
    for candidate, vec in vectors.items():
        if candidate == query or candidate == "<UNK>":
            continue
        denom = q_norm * math.sqrt(sum(value * value for value in vec.values()))
        if denom == 0:
            continue
        # Iterate through the smaller mapping for the dot product.
        small, large = (q, vec) if len(q) <= len(vec) else (vec, q)
        score = sum(value * large.get(context, 0.0) for context, value in small.items()) / denom
        results.append((candidate, score))
    return sorted(results, key=lambda item: item[1], reverse=True)[:k]

queries = ["market", "stock", "bank", "company", "million"]
representations = {
    "PMI (window=2)": ppmi_w2,
    "TF-IDF (sentence docs)": tfidf_sparse,
}

def dense_cosine_neighbors(query, k=5):
    if query not in word2idx:
        return []
    query_idx = word2idx[query]
    scores = W_norm @ W_norm[query_idx]
    scores = scores.copy()
    scores[query_idx] = -np.inf
    scores[word2idx["<UNK>"]] = -np.inf
    best = np.argsort(scores)[::-1][:k]
    return [(idx2word[int(i)], float(scores[i])) for i in best]

for query in queries:
    print(f"\n{query}")
    for name, vectors in representations.items():
        print(f"  {name}: {[(w, round(s, 4)) for w, s in sparse_cosine_neighbors(vectors, query)]}")
    print(f"  Word2Vec: {[(w, round(s, 4)) for w, s in dense_cosine_neighbors(query)]}")

# Approximate storage: sparse coordinate entries or full dense arrays.
def sparse_stats(vectors):
    nnz = sum(len(row) for row in vectors.values())
    return nnz, nnz * (8 + 4)

pmi_nnz, pmi_bytes = sparse_stats(ppmi_w2)
tfidf_nnz, tfidf_bytes = sparse_stats(tfidf_sparse)
embedding_dim = W_target.shape[1]
w2v_params = W_target.size + W_context.size
w2v_bytes = w2v_params * W_target.dtype.itemsize
print("\nApproximate representation storage (excludes Python overhead):")
print(f"PMI: {pmi_nnz:,} nonzeros, about {pmi_bytes / 1024:.1f} KiB as value+index entries")
print(f"TF-IDF: {tfidf_nnz:,} nonzeros, about {tfidf_bytes / 1024:.1f} KiB as value+index entries")
print(f"Word2Vec: {embedding_dim}D target vectors ({W_target.nbytes / 1024:.1f} KiB for exported vectors); both trainable tables use {w2v_bytes / 1024:.1f} KiB")
print(f"Dimensions: PMI context vocabulary = {len(vocab_list):,}; TF-IDF documents = {len(train_tokens_processed):,}; Word2Vec = {embedding_dim}")


market
  PMI (window=2): [('plunge', 0.1332), ('stock', 0.128), ('stock-market', 0.1256), ('the', 0.1137), ('friday', 0.1093)]
  TF-IDF (sentence docs): [('the', 0.2229), ('.', 0.1956), ('stock', 0.1838), (',', 0.1733), ('in', 0.1512)]
  Word2Vec: [('markets', 0.7364), ('car', 0.7133), ('tokyo', 0.7105), ('program', 0.7036), ('fund', 0.7023)]

stock
  PMI (window=2): [('exchange', 0.1891), ('preferred', 0.1782), ('convertible', 0.1333), ('shares', 0.1294), ('market', 0.128)]
  TF-IDF (sentence docs): [('exchange', 0.388), ('composite', 0.3666), ('trading', 0.2805), ('york', 0.2464), ('closed', 0.19)]
  Word2Vec: [('markets', 0.7895), ('activity', 0.7771), ('commodity', 0.767), ('plunge', 0.7591), ('york', 0.7536)]

bank
  PMI (window=2): [('deutsche', 0.1547), ('loan', 0.1399), ('subsidiary', 0.1277), ('ncnb', 0.1116), ('parent', 0.1093)]
  TF-IDF (sentence docs): [('basir', 0.2196), ('tan', 0.2079), ('sri', 0.2079), ('m$', 0.1902), ('bumiputra', 0.1898)]
  Word2Vec: [('buy-out', 0.83

For *company*, PPMI returned local patterns such as *’s* and *its*. TF-IDF returned common words and punctuation such as *the* and the period. Word2Vec retrieved company types and names such as *carrier* and *Merkur*.
For *million*, PPMI and Word2Vec both retrieved *billion*. TF-IDF ranked the dollar sign first, reflecting sentence-level co-occurrence in financial news.

PPMI has about 0.41% populated entries and TF-IDF has about 0.26%. The 50-dimensional Word2Vec export is dense and uses about 3.0 MiB, close to the compact PPMI estimate of 2.9 MiB and above TF-IDF's 1.9 MiB for this corpus. Python dictionary overhead is excluded from the sparse estimates.

## Part 2 · Hidden Markov Model for POS Tagging

We reuse the CoNLL-2000 split above and keep the original case and punctuation because both provide POS cues. The HMM learns which tags start sentences, which tags follow one another, and which words each tag emits. Rare training tokens are pooled into `<UNK>` so we can estimate emission probabilities for unfamiliar words.

Additive smoothing gives every possible outcome a positive probability. The initial cutoff and smoothing strength below are starting points for validation.

In [7]:
from collections import Counter

import numpy as np

hmm_train_tokens = train_dataset["tokens"]
hmm_train_tags = train_dataset["pos_tags"]
hmm_min_word_count = 2
hmm_alpha = 0.1

In [8]:
def fit_pos_hmm(token_sentences, tag_sentences, min_word_count=2, alpha=0.1):
    # Skip empty sentences, which contribute no tag or word counts.
    word_counts = Counter(word for sentence in token_sentences for word in sentence)
    tag_to_id = {
        tag: i for i, tag in enumerate(sorted({tag for sentence in tag_sentences for tag in sentence}))
    }
    known_words = sorted(
        word for word, count in word_counts.items()
        if count >= min_word_count and word != "<UNK>"
    )
    word_to_id = {word: i for i, word in enumerate(["<UNK>", *known_words])}
    unk_id = word_to_id["<UNK>"]
    n_tags = len(tag_to_id)

    initial = np.full(n_tags, alpha, dtype=float)
    transition = np.full((n_tags, n_tags), alpha, dtype=float)
    emission = np.full((n_tags, len(word_to_id)), alpha, dtype=float)

    for words, tags in zip(token_sentences, tag_sentences, strict=True):
        if len(words) != len(tags):
            raise ValueError("Tokens and POS tags must align within each sentence.")
        if not tags:
            continue
        word_ids = [word_to_id.get(word, unk_id) for word in words]
        tag_ids = [tag_to_id[tag] for tag in tags]
        initial[tag_ids[0]] += 1
        np.add.at(transition, (tag_ids[:-1], tag_ids[1:]), 1)
        np.add.at(emission, (tag_ids, word_ids), 1)

    return {
        "tag_to_id": tag_to_id,
        "word_to_id": word_to_id,
        "training_words": set(word_counts),
        "log_initial": np.log(initial / initial.sum()),
        "log_transition": np.log(transition / transition.sum(axis=1, keepdims=True)),
        "log_emission": np.log(emission / emission.sum(axis=1, keepdims=True)),
    }

In [9]:
# Hand-check a corpus with repeated transitions and an unknown emission.
toy_hmm = fit_pos_hmm(
    [["dog", "dog", "dog"], ["runs"], []],
    [["N", "N", "N"], ["V"], []],
    alpha=1.0,
)
noun = toy_hmm["tag_to_id"]["N"]
dog = toy_hmm["word_to_id"]["dog"]
unk = toy_hmm["word_to_id"]["<UNK>"]
assert "runs" not in toy_hmm["word_to_id"]
assert "runs" in toy_hmm["training_words"]
np.testing.assert_allclose(np.exp(toy_hmm["log_initial"][noun]), 0.5)
np.testing.assert_allclose(np.exp(toy_hmm["log_transition"][noun, noun]), 3 / 4)
np.testing.assert_allclose(np.exp(toy_hmm["log_emission"][noun, dog]), 4 / 5)
np.testing.assert_allclose(np.exp(toy_hmm["log_emission"][noun, unk]), 1 / 5)

pos_hmm = fit_pos_hmm(hmm_train_tokens, hmm_train_tags, hmm_min_word_count, hmm_alpha)
for model in (toy_hmm, pos_hmm):
    for key in ("log_initial", "log_transition", "log_emission"):
        log_probs = model[key]
        assert np.isfinite(log_probs).all()
        np.testing.assert_allclose(np.exp(log_probs).sum(axis=-1), 1.0)

print("HMM probability checks passed.")
print(f"Training sentences: {len(hmm_train_tokens):,}")
print(f"Empty training sentences skipped: {sum(not words for words in hmm_train_tokens):,}")
print(f"POS tags: {len(pos_hmm['tag_to_id'])}")
print(f"Emission vocabulary (including <UNK>): {len(pos_hmm['word_to_id']):,}")

HMM probability checks passed.
Training sentences: 8,043
Empty training sentences skipped: 1
POS tags: 44
Emission vocabulary (including <UNK>): 9,111


### Viterbi decoding and evaluation

At each token, Viterbi keeps the best score for every possible ending tag. It adds the previous score, the transition log probability, and the current emission log probability. Backpointers recover the best complete tag sequence. Working in log space avoids multiplying many small probabilities.

We report token accuracy and macro precision, recall, and F1. Macro scores average the per-tag scores equally over tags appearing in the gold or predicted labels for each subset. A tag with an undefined precision or recall receives zero for that measure. The unknown-word subset contains tokens absent from the training text. We also count tokens sent to `<UNK>`, which includes rare training words.

In [10]:
def viterbi_decode(tokens, model, emission_overrides=None):
    if len(tokens) == 0:
        return []

    word_to_id = model["word_to_id"]
    unk_id = word_to_id["<UNK>"]
    word_ids = [word_to_id.get(token, unk_id) for token in tokens]
    transition = model["log_transition"]
    emission = model["log_emission"]
    n_tags = len(model["tag_to_id"])
    backpointers = np.empty((len(tokens), n_tags), dtype=int)

    overrides = emission_overrides or {}
    scores = model["log_initial"] + overrides.get(0, emission[:, word_ids[0]])
    tag_ids = np.arange(n_tags)
    for position, word_id in enumerate(word_ids[1:], start=1):
        # Each column contains the scores of all paths into the current tag.
        candidates = scores[:, None] + transition
        previous = candidates.argmax(axis=0)
        backpointers[position] = previous
        scores = candidates[previous, tag_ids] + overrides.get(position, emission[:, word_id])

    best_path = [int(scores.argmax())]
    for position in range(len(tokens) - 1, 0, -1):
        best_path.append(backpointers[position, best_path[-1]])
    id_to_tag = {i: tag for tag, i in model["tag_to_id"].items()}
    return [id_to_tag[tag_id] for tag_id in reversed(best_path)]

In [11]:
from itertools import product

def toy_path_score(tokens, tags):
    tag_ids = [toy_hmm["tag_to_id"][tag] for tag in tags]
    words = toy_hmm["word_to_id"]
    word_ids = [words.get(token, words["<UNK>"]) for token in tokens]
    score = toy_hmm["log_initial"][tag_ids[0]]
    for position, (word_id, tag_id) in enumerate(zip(word_ids, tag_ids, strict=True)):
        score += toy_hmm["log_emission"][tag_id, word_id]
        if position > 0:
            score += toy_hmm["log_transition"][tag_ids[position - 1], tag_id]
    return score

assert viterbi_decode([], toy_hmm) == []
for sample in (["dog"], ["sleeps"], ["dog", "sleeps", "dog"], ["runs", "dog"]):
    decoded = viterbi_decode(sample, toy_hmm)
    all_paths = product(toy_hmm["tag_to_id"], repeat=len(sample))
    best_score = max(toy_path_score(sample, path) for path in all_paths)
    np.testing.assert_allclose(toy_path_score(sample, decoded), best_score)

print("Viterbi matches exhaustive search on the toy sentences.")

Viterbi matches exhaustive search on the toy sentences.


In [12]:
def pos_metrics(gold, predicted):
    pairs = list(zip(gold, predicted, strict=True))
    if not pairs:
        return dict(tokens=0, accuracy=0.0, precision=0.0, recall=0.0, f1=0.0)

    gold_counts = Counter(gold)
    prediction_counts = Counter(predicted)
    correct = Counter(actual for actual, guess in pairs if actual == guess)
    tags = sorted(gold_counts.keys() | prediction_counts.keys())
    true_positive = np.array([correct[tag] for tag in tags])
    actual_total = np.array([gold_counts[tag] for tag in tags])
    predicted_total = np.array([prediction_counts[tag] for tag in tags])

    precision = true_positive / np.maximum(predicted_total, 1)
    recall = true_positive / np.maximum(actual_total, 1)
    f1 = 2 * true_positive / (actual_total + predicted_total)
    return dict(
        tokens=len(pairs),
        accuracy=float(true_positive.sum() / len(pairs)),
        precision=float(precision.mean()),
        recall=float(recall.mean()),
        f1=float(f1.mean()),
    )


def evaluate_pos_hmm(dataset, model):
    gold, predicted = [], []
    unknown_gold, unknown_predicted = [], []
    mapped_to_unk = 0
    for example in dataset:
        tokens, tags = example["tokens"], example["pos_tags"]
        guesses = viterbi_decode(tokens, model)
        gold.extend(tags)
        predicted.extend(guesses)
        for token, actual, guess in zip(tokens, tags, guesses, strict=True):
            mapped_to_unk += token not in model["word_to_id"]
            if token not in model["training_words"]:
                unknown_gold.append(actual)
                unknown_predicted.append(guess)
    return {
        "overall": pos_metrics(gold, predicted),
        "unknown": pos_metrics(unknown_gold, unknown_predicted),
        "mapped_to_unk": mapped_to_unk,
    }

In [13]:
validation = evaluate_pos_hmm(val_dataset, pos_hmm)

for group in ("overall", "unknown"):
    result = validation[group]
    print(
        f"{group.title()} ({result['tokens']:,} tokens): "
        f"accuracy={result['accuracy']:.2%}, "
        f"precision={result['precision']:.2%}, "
        f"recall={result['recall']:.2%}, F1={result['f1']:.2%}"
    )

print(f"Tokens mapped to <UNK>: {validation['mapped_to_unk']:,}")

Overall (21,369 tokens): accuracy=94.22%, precision=89.17%, recall=85.54%, F1=86.90%
Unknown (1,037 tokens): accuracy=51.88%, precision=25.66%, recall=23.51%, F1=23.46%
Tokens mapped to <UNK>: 1,606


### Select settings on validation

Compare three vocabulary cutoffs and three additive smoothing strengths. Select the model with the highest overall validation macro F1, using unknown-word F1 to break a tie. This choice precedes evaluation on the original test partition. The cutoff controls which rare training tokens are pooled into the unknown-word category.

In [14]:
candidates = []
for min_count in (1, 2, 3):
    for alpha in (0.01, 0.1, 1.0):
        model = fit_pos_hmm(hmm_train_tokens, hmm_train_tags, min_count, alpha)
        result = evaluate_pos_hmm(val_dataset, model)
        candidates.append((result["overall"]["f1"], result["unknown"]["f1"], min_count, alpha))
        print(
            f"cutoff={min_count}, alpha={alpha:g}: "
            f"accuracy={result['overall']['accuracy']:.2%}, "
            f"macro F1={result['overall']['f1']:.2%}, "
            f"unknown F1={result['unknown']['f1']:.2%}"
        )

_, _, best_min_count, best_alpha = max(candidates)
print(f"Selected cutoff={best_min_count}, alpha={best_alpha:g} by validation macro F1.")

cutoff=1, alpha=0.01: accuracy=94.54%, macro F1=82.45%, unknown F1=9.84%
cutoff=1, alpha=0.1: accuracy=94.36%, macro F1=84.32%, unknown F1=12.45%
cutoff=1, alpha=1: accuracy=90.25%, macro F1=76.49%, unknown F1=13.41%
cutoff=2, alpha=0.01: accuracy=94.64%, macro F1=88.61%, unknown F1=24.40%
cutoff=2, alpha=0.1: accuracy=94.22%, macro F1=86.90%, unknown F1=23.46%
cutoff=2, alpha=1: accuracy=90.76%, macro F1=77.50%, unknown F1=21.43%
cutoff=3, alpha=0.01: accuracy=93.67%, macro F1=87.83%, unknown F1=25.69%
cutoff=3, alpha=0.1: accuracy=93.35%, macro F1=86.11%, unknown F1=23.57%
cutoff=3, alpha=1: accuracy=90.34%, macro F1=78.03%, unknown F1=21.26%
Selected cutoff=2, alpha=0.01 by validation macro F1.


In [15]:
final_hmm = fit_pos_hmm(hmm_train_tokens, hmm_train_tags, best_min_count, best_alpha)

## Part 3 · Embedding-assisted HMM

An emission-vocabulary miss can use the embedding of its normalized spelling, such as a lowercase form observed in training. If that spelling has no usable vector, retrieve neighbors of the observed tokens within two positions and pool their cosine weights. Each anchor's contributions sum to its inverse distance. Use positive cosine scores and take the top $k$ candidates after pooling. Each candidate votes with its training tag distribution. Neighbor candidates need labeled occurrences among the HMM's known words.

The unknown tag distribution of the baseline is proportional to $P(\text{UNK}\mid t)P(t)$. We interpolate it with the neighbor votes, then divide by the training tag prior and restore the baseline unknown mass to obtain a position-specific emission score. Known-word emissions, initial probabilities, and transitions use the baseline estimates. With no usable evidence or zero interpolation weight, the emission uses the baseline unknown score.

Choose $k$ and separate interpolation weights for direct and contextual evidence on validation. The primary criterion is accuracy on words absent from training, with overall accuracy as the tiebreaker. A zero contextual weight lets validation disable context-derived votes. The original test data enters after all choices are frozen. The test analysis also tracks changed predictions caused by other positions on the Viterbi path.

In [16]:
from functools import lru_cache

# Estimate P(tag | normalized word) from the tagged training sentences.
tag_counts = np.zeros((len(word2idx), len(final_hmm["tag_to_id"])), dtype=np.float64)
for words, tags in zip(hmm_train_tokens, hmm_train_tags, strict=True):
    for token, tag in zip(words, tags, strict=True):
        word_id = word2idx.get(normalize_token(token))
        if word_id is not None and token in final_hmm["word_to_id"]:
            tag_counts[word_id, final_hmm["tag_to_id"][tag]] += 1

candidate_mask = tag_counts.sum(axis=1) > 0
candidate_mask[word2idx["<UNK>"]] = False
candidate_tags = tag_counts / np.maximum(tag_counts.sum(axis=1, keepdims=True), 1)

# Bayes conversion keeps the unknown column's overall scale while changing its tag preference.
all_tag_counts = Counter(tag for sentence in hmm_train_tags for tag in sentence)
prior = np.array([all_tag_counts[tag] for tag in final_hmm["tag_to_id"]], dtype=float)
prior /= prior.sum()
unknown_emission = np.exp(final_hmm["log_emission"][:, final_hmm["word_to_id"]["<UNK>"]])
unknown_mass = unknown_emission @ prior
baseline_unknown_tags = unknown_emission * prior / unknown_mass

def sparse_similarity(vectors):
    """Return cosine score vectors using inverted postings for normalized sparse rows."""
    rows = []
    postings = defaultdict(list)
    for word_id, word in enumerate(vocab_list):
        row = vectors.get(word, {})
        norm = math.sqrt(sum(value * value for value in row.values()))
        normalized = {column: value / norm for column, value in row.items() if value and norm}
        rows.append(normalized)
        for column, value in normalized.items():
            postings[column].append((word_id, value))

    column_rows = {}
    for column, entries in postings.items():
        ids, weights = zip(*entries)
        column_rows[column] = np.asarray(ids), np.asarray(weights)

    def score(query_id):
        scores = np.zeros(len(vocab_list))
        for column, value in rows[query_id].items():
            ids, weights = column_rows[column]
            scores[ids] += value * weights
        return scores

    return score


def neighbor_lookup(score, limit=40):
    @lru_cache(maxsize=None)
    def neighbors(query_id):
        scores = score(query_id)
        scores[~candidate_mask] = 0.0
        scores[query_id] = 0.0
        positive = np.flatnonzero(scores > 0)
        order = np.lexsort((positive, -scores[positive]))
        best = positive[order[:limit]]
        return tuple(zip(best.tolist(), scores[best].tolist()))

    return neighbors


neighbors_by_system = {
    "PPMI-assisted": neighbor_lookup(sparse_similarity(ppmi_w2)),
    "TF-IDF-assisted": neighbor_lookup(sparse_similarity(tfidf_sparse)),
    "Word2Vec-assisted": neighbor_lookup(lambda query_id: W_norm @ W_norm[query_id]),
}

def pos_evidence(tokens, position, neighbors, top_k):
    """Return a neighbor-weighted tag distribution and the words supporting it."""
    query_id = word2idx.get(normalize_token(tokens[position]))
    pooled = defaultdict(float)
    direct = query_id is not None and query_id != word2idx["<UNK>"]
    if direct:
        pooled.update(neighbors(query_id))

    # An unseen spelling has no learned vector. Nearby in-vocabulary tokens supply proxies.
    if not pooled:
        direct = False
        anchors = []
        for offset in (-2, -1, 1, 2):
            index = position + offset
            if 0 <= index < len(tokens):
                anchor = word2idx.get(normalize_token(tokens[index]))
                if anchor is not None and anchor != word2idx["<UNK>"]:
                    anchors.append((anchor, 1 / abs(offset)))
        for anchor, weight in anchors:
            support = neighbors(anchor)
            total = sum(similarity for _, similarity in support)
            for candidate, similarity in support:
                pooled[candidate] += weight * similarity / total
        for anchor, _ in anchors:
            pooled.pop(anchor, None)

    ranked = sorted(pooled.items(), key=lambda item: (-item[1], item[0]))[:top_k]
    if not ranked:
        return None

    ids, weights = zip(*ranked)
    weights = np.asarray(weights)
    tag_distribution = (weights / weights.sum()) @ candidate_tags[list(ids)]
    supporting_words = tuple(vocab_list[word_id] for word_id in ids[:3])
    return tag_distribution, direct, supporting_words


def decode_with_evidence(tokens, cached_evidence, direct_weight, context_weight):
    overrides = {}
    for position, (estimate, direct, _) in cached_evidence.items():
        weight = direct_weight if direct else context_weight
        if weight:
            mixed = (1 - weight) * baseline_unknown_tags + weight * estimate
            overrides[position] = np.log(unknown_mass * mixed / prior)
    return viterbi_decode(tokens, final_hmm, overrides)

In [17]:
from itertools import product

def prepare_evaluation(dataset):
    rows = []
    for row in dataset:
        tokens, tags = row["tokens"], row["pos_tags"]
        pooled = [i for i, token in enumerate(tokens) if token not in final_hmm["word_to_id"]]
        unseen = [i for i, token in enumerate(tokens) if token not in final_hmm["training_words"]]
        rows.append((tokens, tags, pooled, unseen))
    return rows


def prepare_evidence(rows, neighbors, top_k):
    prepared = []
    for tokens, _, pooled, _ in rows:
        estimates = {}
        for position in pooled:
            evidence = pos_evidence(tokens, position, neighbors, top_k)
            if evidence is not None:
                estimates[position] = evidence
        prepared.append(estimates)
    return prepared


def score_predictions(rows, predictions):
    gold = [tag for _, tags, _, _ in rows for tag in tags]
    predicted = [tag for guesses in predictions for tag in guesses]
    unseen_gold = [tags[i] for _, tags, _, unseen in rows for i in unseen]
    unseen_predicted = [
        guesses[i] for (_, _, _, unseen), guesses in zip(rows, predictions, strict=True) for i in unseen
    ]
    return {"overall": pos_metrics(gold, predicted), "unknown": pos_metrics(unseen_gold, unseen_predicted)}


def predict_assisted(rows, evidence, direct_weight, context_weight):
    return [
        decode_with_evidence(tokens, estimates, direct_weight, context_weight)
        for (tokens, _, _, _), estimates in zip(rows, evidence, strict=True)
    ]


def print_scores(name, scores):
    for group, result in scores.items():
        print(
            f"{name:<20} {group:<8} {result['tokens']:>7,} "
            + " ".join(f"{result[metric]:>9.2%}" for metric in ("accuracy", "precision", "recall", "f1"))
        )


score_header = f"{'System':<20} {'Subset':<8} {'Tokens':>7} {'Accuracy':>9} {'Precision':>9} {'Recall':>9} {'Macro F1':>9}"
validation_rows = prepare_evaluation(val_dataset)
baseline_validation = [viterbi_decode(tokens, final_hmm) for tokens, _, _, _ in validation_rows]
validation_baseline_scores = score_predictions(validation_rows, baseline_validation)
print("Validation baseline")
print(score_header)
print_scores("Baseline HMM", validation_baseline_scores)

selected = {}
for name, neighbors in neighbors_by_system.items():
    best_key = (-1.0, -1.0)
    for top_k in (5, 20):
        evidence = prepare_evidence(validation_rows, neighbors, top_k)
        for direct_weight, context_weight in product((0.35, 0.7), (0.0, 0.15, 0.35)):
            guesses = predict_assisted(validation_rows, evidence, direct_weight, context_weight)
            scores = score_predictions(validation_rows, guesses)
            key = scores["unknown"]["accuracy"], scores["overall"]["accuracy"]
            print(
                f"{name:<20} k={top_k:>2} direct={direct_weight:.2f} context={context_weight:.2f} "
                f"unknown accuracy={key[0]:.2%} overall accuracy={key[1]:.2%}"
            )
            if key > best_key:
                best_key = key
                selected[name] = (top_k, direct_weight, context_weight)
    print(f"Selected {name}: k, direct weight, context weight = {selected[name]}")

# Zero interpolation must recover the baseline, including when queried evidence exists.
sample_evidence = prepare_evidence(validation_rows[:10], neighbors_by_system["PPMI-assisted"], 5)
assert predict_assisted(validation_rows[:10], sample_evidence, 0.0, 0.0) == baseline_validation[:10]

Validation baseline
System               Subset    Tokens  Accuracy Precision    Recall  Macro F1
Baseline HMM         overall   21,369    94.64%    89.99%    88.36%    88.61%
Baseline HMM         unknown    1,037    52.46%    26.48%    24.54%    24.40%
PPMI-assisted        k= 5 direct=0.35 context=0.00 unknown accuracy=54.58% overall accuracy=94.80%
PPMI-assisted        k= 5 direct=0.35 context=0.15 unknown accuracy=52.46% overall accuracy=94.67%
PPMI-assisted        k= 5 direct=0.35 context=0.35 unknown accuracy=50.63% overall accuracy=94.53%
PPMI-assisted        k= 5 direct=0.70 context=0.00 unknown accuracy=54.39% overall accuracy=94.77%
PPMI-assisted        k= 5 direct=0.70 context=0.15 unknown accuracy=52.07% overall accuracy=94.65%
PPMI-assisted        k= 5 direct=0.70 context=0.35 unknown accuracy=50.63% overall accuracy=94.52%
PPMI-assisted        k=20 direct=0.35 context=0.00 unknown accuracy=54.77% overall accuracy=94.80%
PPMI-assisted        k=20 direct=0.35 context=0.15 un

### Held-out results and error examples

In [18]:
test_rows = prepare_evaluation(test_dataset)
test_predictions = {
    "Baseline HMM": [viterbi_decode(tokens, final_hmm) for tokens, _, _, _ in test_rows]
}
test_evidence = {}
test_scores = {"Baseline HMM": score_predictions(test_rows, test_predictions["Baseline HMM"])}
print(score_header)
print_scores("Baseline HMM", test_scores["Baseline HMM"])
for name, neighbors in neighbors_by_system.items():
    top_k, direct_weight, context_weight = selected[name]
    evidence = prepare_evidence(test_rows, neighbors, top_k)
    test_evidence[name] = evidence
    guesses = predict_assisted(test_rows, evidence, direct_weight, context_weight)
    test_predictions[name] = guesses
    scores = score_predictions(test_rows, guesses)
    test_scores[name] = scores
    print_scores(name, scores)

print("\nUnknown-word evidence and changes relative to baseline")
baseline = test_predictions["Baseline HMM"]
tag_label = test_dataset.features["pos_tags"].feature.int2str
ranked_examples = {}
for name, evidence in test_evidence.items():
    direct_count = context_count = rescued = harmed = 0
    cases = {"rescued": [], "harmed": []}
    guesses = test_predictions[name]
    _, direct_weight, context_weight = selected[name]
    for sentence_id, ((tokens, tags, _, unseen), estimates) in enumerate(zip(test_rows, evidence, strict=True)):
        for position in unseen:
            entry = estimates.get(position)
            active = entry is not None and (direct_weight if entry[1] else context_weight) > 0
            if active:
                _, direct, supporters = entry
                direct_count += int(direct)
                context_count += int(not direct)
                evidence_type = "direct" if direct else "context"
            else:
                supporters = ()
                evidence_type = "sequence"
            actual = tags[position]
            old = baseline[sentence_id][position]
            new = guesses[sentence_id][position]
            rescued += int(old != actual and new == actual)
            harmed += int(old == actual and new != actual)
            if (old == actual) != (new == actual):
                kind = "rescued" if new == actual else "harmed"
                cases[kind].append((tokens, position, actual, old, new, evidence_type, supporters))
    print(f"{name:<20} direct={direct_count:,} context={context_count:,} corrected={rescued:,} regressed={harmed:,}")
    for kind, items in cases.items():
        # Show one example from each evidence source, when available.
        for source in ("direct", "context", "sequence"):
            item = next((item for item in items if item[5] == source), None)
            if item is None:
                continue
            tokens, position, actual, old, new, evidence_type, supporters = item
            start = max(0, position - 4)
            end = min(len(tokens), position + 5)
            display_tokens = tokens[start:end].copy()
            display_tokens[position - start] = "[" + tokens[position] + "]"
            print(
                f"  {kind:<7} {tag_label(old)} → {tag_label(new)}, gold={tag_label(actual)} "
                f"({' '.join(display_tokens)}), {evidence_type}: {', '.join(supporters)}"
            )

System               Subset    Tokens  Accuracy Precision    Recall  Macro F1
Baseline HMM         overall   47,377    93.43%    92.91%    90.44%    91.05%
Baseline HMM         unknown    3,438    54.77%    21.51%    20.24%    20.16%
PPMI-assisted        overall   47,377    93.59%    92.87%    90.54%    91.08%
PPMI-assisted        unknown    3,438    56.34%    21.84%    20.98%    20.65%
TF-IDF-assisted      overall   47,377    93.44%    92.76%    90.51%    91.01%
TF-IDF-assisted      unknown    3,438    54.83%    18.95%    17.48%    17.71%
Word2Vec-assisted    overall   47,377    93.48%    92.92%    90.49%    91.08%
Word2Vec-assisted    unknown    3,438    55.12%    30.66%    25.58%    27.02%

Unknown-word evidence and changes relative to baseline
PPMI-assisted        direct=535 context=0 corrected=93 regressed=39
  rescued NNP → CD, gold=CD (in either 1966 or [1967] , intelligence officials say), direct: %, $, million
  harmed  NNP → NN, gold=NNP (book review `` Kissing [Nature] Good-

### Appendix · Baseline per-tag test results

In [19]:
# Count actual and predicted POS tag pairs on the held-out test set.
tag_ids = final_hmm["tag_to_id"]
confusion = np.zeros((len(tag_ids), len(tag_ids)), dtype=int)

for (_, tags, _, _), predictions in zip(test_rows, baseline, strict=True):
    for actual, predicted in zip(tags, predictions, strict=True):
        confusion[tag_ids[actual], tag_ids[predicted]] += 1

total = confusion.sum()
tp = confusion.diagonal()
fp = confusion.sum(axis=0) - tp
fn = confusion.sum(axis=1) - tp
tn = total - tp - fp - fn
f1_denominator = 2 * tp + fp + fn
active_tags = f1_denominator > 0
tag_f1 = 2 * tp / np.maximum(f1_denominator, 1)

print("Per-tag one-vs-rest confusion counts")
print(f"{'Tag':<6} {'TP':>8} {'FP':>8} {'FN':>8} {'TN':>8} {'F1':>8}")
tag_labels = test_dataset.features["pos_tags"].feature
for tag, i in sorted(tag_ids.items(), key=lambda item: item[1]):
    label = tag_labels.int2str(tag)
    score = f"{tag_f1[i]:.2%}" if active_tags[i] else "—"
    print(f"{label:<6} {tp[i]:>8,} {fp[i]:>8,} {fn[i]:>8,} {tn[i]:>8,} {score:>8}")

correct = tp.sum()
incorrect = total - correct
accuracy = correct / total
macro_f1 = tag_f1[active_tags].mean()
np.testing.assert_allclose(
    [total, accuracy, macro_f1],
    [test_scores["Baseline HMM"]["overall"][key] for key in ("tokens", "accuracy", "f1")],
)

print()
print("Overall held-out test results")
print(f"{'Correct':>9} {'Incorrect':>9} {'Total':>9} {'Accuracy':>9} {'Macro F1':>9}")
print(f"{correct:>9,} {incorrect:>9,} {total:>9,} {accuracy:>9.2%} {macro_f1:>9.2%}")


Per-tag one-vs-rest confusion counts
Tag          TP       FP       FN       TN       F1
''          314        2        2   47,059   99.37%
#            11        0        0   47,366  100.00%
$           384       11        0   46,982   98.59%
(            77        0        0   47,300  100.00%
)            77        0        0   47,300  100.00%
,         2,390        0        0   44,987  100.00%
.         1,975        4        0   45,398   99.90%
:           238        2        0   47,137   99.58%
``          323        1        0   47,053   99.85%
CC        1,212        0        2   46,163   99.92%
CD        1,734      183      184   45,276   90.43%
DT        3,996       21       24   43,336   99.44%
EX           47        2        1   47,327   96.91%
FW            2        1        2   47,372   57.14%
IN        4,999       85       72   42,221   98.45%
JJ        2,683      655      281   43,758   85.15%
JJR         190       14       12   47,161   93.60%
JJS          70        1   

### Findings

PPMI had the best test accuracy overall and on the 3,438 unknown tokens. It corrected 93 unknown-word predictions and introduced 39 errors, improving unknown accuracy from 54.77% to 56.34%. Word2Vec reached the highest reported unknown macro F1, 27.02% versus 20.16% for the baseline. Its unknown accuracy improved by only 0.35 percentage points. TF-IDF changed unknown accuracy by 0.06 percentage points and reduced unknown macro F1 to 17.71%. Macro F1 weights each active tag equally, and the active tag set is computed from the gold and predicted tags for each system and subset.

All three systems selected 20 neighbors and a direct interpolation weight of 0.70. Validation selected zero contextual weight in all three cases. For the chosen PPMI settings, adding a contextual weight of 0.15 lowered validation unknown accuracy from 55.26% to 54.10%. Direct evidence changed emission scores for 535 of 3,438 truly unseen test tokens (15.56%). The others kept their baseline unknown emissions, although Viterbi could still change their predicted tags when another position received a different score.

- PPMI corrected **1967** from proper noun (`NNP`) to numeral (`CD`). Its normalized form retrieved number-associated words such as *million* and the percent sign. PPMI mistagged **Nature** in the book title *Kissing Nature Good-bye* as a common noun (`NN`). Lowercasing removed the proper-name cue and returned neighbors such as *correction*.
- TF-IDF also corrected **1967**, but changed **Baby** in *Baby Doc Duvalier* from the correct proper noun to an adjective. Neighbors such as *boomers* and *aging* reflect sentence associations of the lowercase word. The changed scores also shifted **Doc** to an adjective through the Viterbi path.
- Word2Vec corrected **Enormous** from proper noun to adjective (`JJ`). Adjective neighbors such as *flashy* supplied useful evidence despite the initial capitalization. The new path also corrected the following **ice** to common noun. The same lowercase lookup hurt *Nature*, where preserving capitalization was important.

The results favor local PPMI evidence for token accuracy and dense Word2Vec evidence for the reported balance across unknown POS tags. The coverage depends on recovering a trained representation through normalization. The tested contextual approximation gave weaker validation results for spellings without a usable vector.